# Fine-tune on GPU

One-click training on **Colab**, **Vertex AI Workbench**, or any NVIDIA GPU environment.

| GPU | VRAM | Max Model | Batch | Seq |
|-----|------|-----------|-------|-----|
| T4 | 16 GB | 7B-4bit | 1 | 2048 |
| L4 | 24 GB | 14B-4bit | 1 | 2048 |
| A100 40GB | 40 GB | 32B-4bit | 1 | 4096 |
| A100 80GB | 80 GB | 32B-4bit | 4 | 4096 |

In [ ]:
#@title Setup { display-mode: "form" }
#@markdown Clone repo and install Unsloth. Takes ~3 min, no kernel restart needed.
#@markdown On Colab Enterprise (GCE), waits for NVIDIA drivers to finish installing.

import os, subprocess, time, importlib.util

# — Clone / update —
REPO = "https://github.com/piotrjanik/ai-tuner.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
if not os.path.exists("ai-tuner"):
    !git clone --depth 1 --branch {BRANCH} {REPO}
%cd ai-tuner
!git checkout {BRANCH}
!git pull origin {BRANCH}

# — Wait for NVIDIA drivers (GCE Deep Learning VMs install on first boot) —
print("Checking for NVIDIA GPU drivers...")
for i in range(30):  # wait up to 5 min
    r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    if r.returncode == 0:
        print(r.stdout.split(chr(10))[2])  # GPU name line
        break
    if i == 0:
        print("  Drivers still installing (normal on first boot, up to 5 min)...")
    if i % 6 == 5:
        print(f"  Still waiting... ({(i+1)*10}s)")
    time.sleep(10)
else:
    print("❌ NVIDIA drivers not found after 5 min.")
    print("  If using Colab Enterprise:")
    print("    1. Check your Runtime Template has GPU accelerator (NVIDIA L4)")
    print("    2. Machine type must match GPU (g2-standard-8 for L4)")
    print("    3. Try: Runtime > Disconnect and delete runtime, then reconnect")
    raise RuntimeError("No NVIDIA GPU drivers found")

# — Install Unsloth + deps (--no-deps for unsloth/trl to keep CUDA torch) —
!pip install -q --no-deps \
    "unsloth @ git+https://github.com/unslothai/unsloth" \
    "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo"
!pip install -q --no-deps trl==0.22.2
!pip install -q transformers==4.56.2 tokenizers bitsandbytes triton \
    "peft>=0.13.0" "accelerate>=1.0.0" "typer>=0.12.0" \
    pyyaml tqdm "huggingface_hub[cli]" "datasets>=2.14.0" xformers

# — Flash Attention (optional, Ampere+ only) —
!timeout 60 pip install -q flash-attn --no-build-isolation 2>/dev/null || true

# — Verify —
!python -c "import torch; print(f'torch {torch.__version__}, CUDA={torch.cuda.is_available()}")"
!python -c "import unsloth; print(f'unsloth OK')"

In [ ]:
# — GPU check —
import subprocess, torch

# Check nvidia-smi first
r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if r.returncode != 0:
    print("❌ nvidia-smi failed! No GPU visible to the runtime.")
    print("   → Go to Runtime > Change runtime type > select GPU (L4 or A100)")
    print("   → Then click 'Save' and re-run all cells.")
else:
    print(r.stdout.split(chr(10))[0:9])

print(f"torch {torch.__version__}, cuda.is_available={torch.cuda.is_available()}")

if not torch.cuda.is_available() and "+cu" in torch.__version__:
    print("❌ Torch has CUDA bindings but CUDA is not available.")
    print("   Possible causes:")
    print("   1. GPU runtime not selected (Runtime > Change runtime type > GPU)")
    print("   2. Runtime just started — try: Runtime > Restart runtime, then re-run")
    print("   3. Driver mismatch — check nvidia-smi output above")

assert torch.cuda.is_available(), "No CUDA GPU detected. See diagnostics above."
props = torch.cuda.get_device_properties(0)
print(f"✓ GPU: {props.name} ({props.total_memory / 1e9:.0f} GB)")
fa2 = "yes" if props.major >= 8 else "no (eager fallback)"
print(f"✓ Flash Attention 2: {fa2}")

In [ ]:
#@title Prepare data & train { display-mode: "form" }
#@markdown Extracts training data from config.yaml sources, then runs Unsloth QLoRA training.

!python src/data/prepare_data.py --config config.yaml
!python src/train/train_cuda.py --config config.yaml

In [ ]:
# (Optional) Push adapters to HuggingFace Hub
# !python src/cli.py push